# YOLOv8s AFPN + ATFL + Inner-IoU + Sample-Aware Box Weighting

This notebook is for the clean 3-class experiment only.

Fixed settings in this notebook:
- model: `yolov8s_afpn.yaml`
- cls loss: `ATFL`
- box loss: `Inner-IoU`
- regression weighting: `sample-aware box/DFL weighting`
- classes: `chuck`, `gripper`, `drill_pipe`

Notes:
- The notebook writes a fresh 3-class `data.yaml`, so it will not accidentally train as 4 classes.
- `yolov8s.pt` is downloaded automatically if missing.


In [ ]:
# Cell 1: clone repo and install local package
%cd /kaggle/working
!rm -rf yolov8s_kuangjing
!git clone https://github.com/tuanziya666/yolov8s_kuangjing.git
%cd /kaggle/working/yolov8s_kuangjing
!python -m pip uninstall -y ultralytics >/dev/null 2>&1 || true
!python -m pip install -q -e .
!python -c "import ultralytics, sys; print('python=', sys.executable); print('ultralytics=', ultralytics.__file__)"

In [ ]:
# Cell 2: dataset yaml for the clean 3-class setup
from pathlib import Path

CANDIDATE_ROOTS = [
    "/kaggle/input/yolo-datasets-10k-yolov8-3class",
    "/kaggle/input/yolo-datasets-10k-yolov8-3class/yolo_datasets_10k_yolov8_3class",
    "/kaggle/input/re-new-datasets/yolo_datasets_10k_yolov8_3class",
    "/kaggle/input/your-3class-dataset",
]

DATASET_ROOT = next((p for p in CANDIDATE_ROOTS if Path(p).exists()), CANDIDATE_ROOTS[-1])
DATASET_ROOT = Path(DATASET_ROOT)
print('DATASET_ROOT =', DATASET_ROOT)
if not DATASET_ROOT.exists():
    raise FileNotFoundError('Please update DATASET_ROOT to your Kaggle dataset path before training.')

DATA_CFG = Path('/kaggle/working/yolov8s_kuangjing/ultralytics/cfg/datasets/coalmine3_kaggle.yaml')
yaml_text = f"""path: {DATASET_ROOT.as_posix()}
train: images/train
val: images/val
test: images/test

names:
  0: chuck
  1: gripper
  2: drill_pipe
"""
DATA_CFG.write_text(yaml_text, encoding='utf-8')
print(DATA_CFG.read_text(encoding='utf-8'))

In [ ]:
# Cell 3: common setup
import random
import numpy as np
import torch
import requests
from pathlib import Path
from PIL import Image
from ultralytics.utils.downloads import attempt_download_asset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

EPOCHS = 300
BATCH = 16
IMGSZ = 640
WORKERS = 2
DEVICE = '0'
PROJECT = '/kaggle/working/runs/detect'
NAME = 'yolov8s_afpn_atfl_inneriou_sa_box_3cls_seed42_e300'

SA_SMALL_ALPHA = 0.5
SA_ELONG_BETA = 0.7
SA_CLASS_GAMMA = 0.5
SA_SMALL_AREA_THR = 0.012521
SA_ELONG_RATIO_THR = 3.0
SA_TARGET_CLASS_IDS = '2'

assets_dir = Path('/kaggle/working/yolov8s_kuangjing/ultralytics/assets')
assets_dir.mkdir(parents=True, exist_ok=True)

def ensure_valid_image(path: Path, url: str) -> None:
    valid = False
    if path.exists():
        try:
            with Image.open(path) as im:
                im.verify()
            valid = True
        except Exception:
            path.unlink(missing_ok=True)
    if not valid:
        response = requests.get(url, timeout=60)
        response.raise_for_status()
        path.write_bytes(response.content)
        with Image.open(path) as im:
            im.verify()

ensure_valid_image(assets_dir / 'bus.jpg', 'https://ultralytics.com/images/bus.jpg')
ensure_valid_image(assets_dir / 'zidane.jpg', 'https://ultralytics.com/images/zidane.jpg')
weights_path = attempt_download_asset('yolov8s.pt')
print('weights_path =', weights_path)
print('PROJECT =', PROJECT)
print('NAME =', NAME)

In [ ]:
# Cell 4: launch training through the dedicated script
%cd /kaggle/working/yolov8s_kuangjing

import subprocess
import sys

cmd = [
    sys.executable,
    '-u',
    'train_yolov8s_afpn_atfl_inneriou_sa_box.py',
    '--data', str(DATA_CFG),
    '--model', 'ultralytics/cfg/models/v8/yolov8s_afpn.yaml',
    '--pretrained', 'yolov8s.pt',
    '--epochs', str(EPOCHS),
    '--batch', str(BATCH),
    '--imgsz', str(IMGSZ),
    '--workers', str(WORKERS),
    '--device', DEVICE,
    '--optimizer', 'SGD',
    '--seed', str(SEED),
    '--deterministic', 'True',
    '--cache', 'False',
    '--amp', 'True',
    '--project', PROJECT,
    '--name', NAME,
    '--inner-ratio', '0.8',
    '--sa-small-alpha', str(SA_SMALL_ALPHA),
    '--sa-elong-beta', str(SA_ELONG_BETA),
    '--sa-class-gamma', str(SA_CLASS_GAMMA),
    '--sa-small-area-thr', str(SA_SMALL_AREA_THR),
    '--sa-elong-ratio-thr', str(SA_ELONG_RATIO_THR),
    '--sa-target-class-ids', SA_TARGET_CLASS_IDS,
]

print('Running command:')
print(' '.join(cmd))
subprocess.run(cmd, check=True)

In [ ]:
# Cell 5: inspect outputs
from pathlib import Path
import pandas as pd

run_dir = Path(PROJECT) / NAME
print('run_dir =', run_dir)
print('results.csv exists =', (run_dir / 'results.csv').exists())
print('best.pt exists =', (run_dir / 'weights' / 'best.pt').exists())
print('last.pt exists =', (run_dir / 'weights' / 'last.pt').exists())

if (run_dir / 'results.csv').exists():
    df = pd.read_csv(run_dir / 'results.csv')
    tracked_cols = [c for c in df.columns if c.startswith('metrics/') or c.startswith('sa/')]
    display(df[['epoch'] + tracked_cols].tail())

In [ ]:
# Cell 6: package artifacts for download
from pathlib import Path
import shutil

run_dir = Path(PROJECT) / NAME
if not run_dir.exists():
    raise FileNotFoundError(f'Run directory not found: {run_dir}')

full_zip = Path('/kaggle/working') / f'{NAME}_full'
weights_zip = Path('/kaggle/working') / f'{NAME}_weights_only'

full_zip_path = shutil.make_archive(str(full_zip), 'zip', root_dir=run_dir)
print('full_zip_path =', full_zip_path)

tmp_dir = Path('/kaggle/working') / f'{NAME}_weights_only_tmp'
if tmp_dir.exists():
    shutil.rmtree(tmp_dir)
tmp_dir.mkdir(parents=True, exist_ok=True)

for rel_path in [
    'weights/best.pt',
    'weights/last.pt',
    'results.csv',
    'args.yaml',
    'results.png',
    'confusion_matrix.png',
    'confusion_matrix_normalized.png',
    'F1_curve.png',
    'P_curve.png',
    'R_curve.png',
    'PR_curve.png',
]:
    src = run_dir / rel_path
    if src.exists():
        dst = tmp_dir / rel_path
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, dst)

weights_zip_path = shutil.make_archive(str(weights_zip), 'zip', root_dir=tmp_dir)
print('weights_zip_path =', weights_zip_path)

shutil.rmtree(tmp_dir, ignore_errors=True)